# B1 - Normalizacion Silver

Este notebook lee los datos limpios de `data/silver/cleaned/` y genera datasets normalizados en `data/silver/trip_data_normalized/` usando PySpark.

La normalizacion genera una salida ligera con columnas canonicas necesarias para Gold, dashboards y modelos: tipo de dataset, anio, ubicaciones pickup/dropoff, zonas, duracion, distancia, importes, pago y pasajeros. Evita duplicar columnas originales cuando ya existe una version canonica.

## 1. Configuracion del entorno

In [1]:
from pathlib import Path
from datetime import datetime
import gc
import json
import os
import re
import subprocess
import urllib.request


def find_project_root(start=None) -> Path:
    """Busca la raiz del proyecto aunque el notebook se ejecute desde otra carpeta."""
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        cleaned = candidate / "data" / "silver" / "cleaned"
        if cleaned.exists():
            return candidate
    raise FileNotFoundError("No se encontro la raiz del proyecto con data/silver/cleaned.")


PROJECT_ROOT = find_project_root()
CLEANED_BASE_DIR = PROJECT_ROOT / "data" / "silver" / "cleaned"
OUTPUT_BASE_DIR = PROJECT_ROOT / "data" / "silver" / "trip_data_normalized"
LOG_DIR = PROJECT_ROOT / "data" / "logs"
AUDIT_LOG_PATH = LOG_DIR / "silver_b_normalization_audit.jsonl"

OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Spark en Windows necesita winutils.exe y hadoop.dll antes de crear SparkSession.
# Esta configuracion debe ejecutarse con el kernel recien iniciado.
SYSTEM_HADOOP_HOME = Path("C:/hadoop")
LOCAL_HADOOP_HOME = (
    SYSTEM_HADOOP_HOME
    if SYSTEM_HADOOP_HOME.exists() or os.access(SYSTEM_HADOOP_HOME.parent, os.W_OK)
    else Path.home() / ".hadoop"
).resolve()
HADOOP_HOME_JAVA = LOCAL_HADOOP_HOME.as_posix()
LOCAL_HADOOP_BIN = LOCAL_HADOOP_HOME / "bin"
LOCAL_HADOOP_BIN.mkdir(parents=True, exist_ok=True)

WINUTILS_BASE_URL = "https://raw.githubusercontent.com/steveloughran/winutils/master/hadoop-3.0.0/bin/"
for hadoop_file in ["winutils.exe", "hadoop.dll"]:
    target = LOCAL_HADOOP_BIN / hadoop_file
    if not target.exists():
        print(f"Descargando {hadoop_file} para PySpark en Windows...")
        try:
            urllib.request.urlretrieve(WINUTILS_BASE_URL + hadoop_file, target)
        except Exception as error:
            raise RuntimeError(
                f"No se pudo descargar {hadoop_file}. Descargalo manualmente en {LOCAL_HADOOP_BIN}. Error: {error}"
            )

os.environ["HADOOP_HOME"] = HADOOP_HOME_JAVA
os.environ["hadoop.home.dir"] = HADOOP_HOME_JAVA
os.environ["PATH"] = str(LOCAL_HADOOP_BIN) + os.pathsep + os.environ.get("PATH", "")


def java_major_version() -> int | None:
    try:
        result = subprocess.run(["java", "-version"], capture_output=True, text=True, check=False)
        version_text = result.stderr or result.stdout
    except Exception:
        return None
    match = re.search(r'version "(\d+)', version_text)
    return int(match.group(1)) if match else None


def find_jdk17_home() -> Path | None:
    search_roots = [
        Path("C:/Program Files/Eclipse Adoptium"),
        Path("C:/Program Files/Java"),
        Path("C:/Program Files/Microsoft"),
        Path("C:/Program Files/Oracle"),
    ]
    for root in search_roots:
        if not root.exists():
            continue
        for candidate in root.glob("**/jdk-17*"):
            if (candidate / "bin" / "java.exe").exists():
                return candidate
    return None


current_java = java_major_version()
if current_java is not None and current_java > 17:
    jdk17_home = find_jdk17_home()
    if jdk17_home is None:
        raise RuntimeError(
            f"PySpark no esta arrancando porque tu Java actual es {current_java}. "
            "Instala JDK 17 y reinicia VS Code/Jupyter. Recomendado: Temurin/OpenJDK 17."
        )
    os.environ["JAVA_HOME"] = str(jdk17_home)
    os.environ["PATH"] = str(jdk17_home / "bin") + os.pathsep + os.environ.get("PATH", "")
    print(f"JAVA_HOME ajustado a JDK 17: {jdk17_home}")
else:
    print(f"Java detectado: {current_java}")

print(f"Proyecto: {PROJECT_ROOT}")
print(f"Entrada cleaned: {CLEANED_BASE_DIR}")
print(f"Salida normalizada: {OUTPUT_BASE_DIR}")
print(f"Auditoria: {AUDIT_LOG_PATH}")
print(f"HADOOP_HOME: {os.environ['HADOOP_HOME']}")

Java detectado: 17
Proyecto: C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos
Entrada cleaned: C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos\data\silver\cleaned
Salida normalizada: C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos\data\silver\trip_data_normalized
Auditoria: C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos\data\logs\silver_b_normalization_audit.jsonl
HADOOP_HOME: C:/hadoop


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import DataFrame

# Refuerza la propiedad dentro de la JVM cuando Spark arranca.
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--driver-memory 4g "
    f"--conf spark.driver.extraJavaOptions=-Dhadoop.home.dir={HADOOP_HOME_JAVA} "
    f"--conf spark.executor.extraJavaOptions=-Dhadoop.home.dir={HADOOP_HOME_JAVA} "
    "pyspark-shell"
)

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("B1_Silver_Normalization_TLC")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "2g")
    .config("spark.driver.maxResultSize", "512m")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "false")
    .config("spark.sql.shuffle.partitions", "128")
    .config("spark.sql.files.maxPartitionBytes", str(64 * 1024 * 1024))
    .config("spark.sql.parquet.columnarReaderBatchSize", "1024")
    .config("spark.driver.extraJavaOptions", f"-Dhadoop.home.dir={HADOOP_HOME_JAVA}")
    .config("spark.executor.extraJavaOptions", f"-Dhadoop.home.dir={HADOOP_HOME_JAVA}")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.sparkContext._jvm.java.lang.System.setProperty("hadoop.home.dir", HADOOP_HOME_JAVA)
spark.sparkContext._jvm.java.lang.System.setProperty("HADOOP_HOME", HADOOP_HOME_JAVA)
print("JVM hadoop.home.dir:", spark.sparkContext._jvm.java.lang.System.getProperty("hadoop.home.dir"))
print("Spark master:", spark.sparkContext.master)
print("Spark driver memory:", spark.sparkContext.getConf().get("spark.driver.memory"))
spark

JVM hadoop.home.dir: C:/hadoop
Spark master: local[2]
Spark driver memory: 4g


## 2. Funciones auxiliares

In [3]:
def clean_column_name(name: str) -> str:
    name = name.strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return re.sub(r"_+", "_", name).strip("_")


def quote_column(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"


def normalize_columns(df: DataFrame) -> DataFrame:
    """Normaliza nombres a snake_case sin descartar columnas."""
    used_names = set()
    aliases = []
    for original_name in df.columns:
        base_name = clean_column_name(original_name)
        new_name = base_name
        suffix = 2
        while new_name in used_names:
            new_name = f"{base_name}_{suffix}"
            suffix += 1
        used_names.add(new_name)
        aliases.append(F.col(quote_column(original_name)).alias(new_name))
    return df.select(*aliases)


def first_existing(columns, candidates):
    available = set(columns)
    for candidate in candidates:
        if candidate in available:
            return candidate
    return None


def column_expr_or_null(df: DataFrame, candidates, data_type: str):
    source_column = first_existing(df.columns, candidates)
    if source_column is None:
        return F.lit(None).cast(data_type), None
    return F.col(source_column).cast(data_type), source_column


def ensure_canonical_column(df: DataFrame, name: str, expr, created_columns: list[str]) -> DataFrame:
    if name in df.columns:
        return df.withColumn(name, F.col(name))
    created_columns.append(name)
    return df.withColumn(name, expr)


def parquet_part_files(path: Path):
    """Devuelve archivos Parquet concretos para evitar errores de listado Hadoop/Windows."""
    if path.is_file() and path.suffix.lower() == ".parquet":
        return [str(path)]
    files = sorted(file for file in path.glob("*.parquet") if file.is_file())
    if not files:
        files = sorted(file for file in path.rglob("*.parquet") if file.is_file())
    return [str(file) for file in files]


def parse_cleaned_dataset_path(path: Path):
    """Extrae tipo y anio desde rutas como cleaned/yellow/year=2023/yellow_2023_cleaned.parquet."""
    taxi_type = path.parent.parent.name.lower()
    year_match = re.match(r"year=(\d{4})$", path.parent.name.lower())
    year = int(year_match.group(1)) if year_match else None
    name_match = re.match(r"([a-z0-9]+)_(\d{4})_cleaned\.parquet$", path.name.lower())
    if name_match:
        taxi_type = name_match.group(1)
        year = int(name_match.group(2))
    return taxi_type, year


def append_audit(record: dict) -> None:
    record = {
        "pipeline": "silver_b_normalization",
        "executed_at": datetime.now().isoformat(timespec="seconds"),
        **record,
    }
    with AUDIT_LOG_PATH.open("a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")


def count_records(df: DataFrame) -> int:
    return int(df.count())


def release_spark_memory() -> None:
    gc.collect()
    try:
        spark.catalog.clearCache()
        spark.sparkContext._jvm.java.lang.System.gc()
    except Exception:
        pass

## 3. Reglas de columnas canonicas

In [4]:
PICKUP_DATETIME_CANDIDATES = ["pickup_datetime", "tpep_pickup_datetime", "lpep_pickup_datetime"]
DROPOFF_DATETIME_CANDIDATES = ["dropoff_datetime", "tpep_dropoff_datetime", "lpep_dropoff_datetime"]
PICKUP_LOCATION_CANDIDATES = ["pickup_location_id", "pulocationid", "pu_location_id", "pu_locationid"]
DROPOFF_LOCATION_CANDIDATES = ["dropoff_location_id", "dolocationid", "do_location_id", "do_locationid"]
DISTANCE_CANDIDATES = ["distancia_millas", "trip_distance", "distance_miles"]
BASE_AMOUNT_CANDIDATES = ["monto_base", "fare_amount", "base_passenger_fare"]
TOTAL_AMOUNT_CANDIDATES = ["monto_total", "total_amount", "base_passenger_fare"]
TIP_CANDIDATES = ["propina", "tip_amount", "tips"]
TOLLS_CANDIDATES = ["peajes", "tolls_amount", "tolls"]
PAYMENT_CANDIDATES = ["payment_type", "paymenttype"]
PASSENGER_CANDIDATES = ["passenger_count", "passengercount"]
VENDOR_CANDIDATES = ["vendor_id", "vendorid"]
RATECODE_CANDIDATES = ["ratecode_id", "ratecodeid"]
SHARED_RIDE_CANDIDATES = ["shared_ride_flag", "sr_flag"]

OPTIONAL_DOUBLE_COLUMNS = {
    "extra": ["extra"],
    "mta_tax": ["mta_tax"],
    "improvement_surcharge": ["improvement_surcharge"],
    "congestion_surcharge": ["congestion_surcharge"],
    "airport_fee": ["airport_fee", "airportfee"],
    "cbd_congestion_fee": ["cbd_congestion_fee"],
    "ehail_fee": ["ehail_fee"],
}

CANONICAL_ORDER = [
    "tipo_dataset",
    "anio",
    "pickup_datetime",
    "dropoff_datetime",
    "pickup_location_id",
    "dropoff_location_id",
    "distancia_millas",
    "duracion_segundos",
    "duracion_minutos",
    "monto_base",
    "monto_total",
    "propina",
    "peajes",
    "extra",
    "mta_tax",
    "improvement_surcharge",
    "congestion_surcharge",
    "airport_fee",
    "cbd_congestion_fee",
    "ehail_fee",
    "payment_type",
    "passenger_count",
    "vendor_id",
    "ratecode_id",
    "trip_type",
    "shared_ride_flag",
    "pickup_date",
    "pickup_hour",
    "pickup_day_of_week",
    "pickup_month",
]

ESSENTIAL_EXTRA_COLUMNS = [
    "pickup_zone",
    "pickup_borough",
    "pickup_service_zone",
    "dropoff_zone",
    "dropoff_borough",
    "dropoff_service_zone",
    "store_and_fwd_flag",
    "dispatching_base_num",
    "affiliated_base_number",
]

DUPLICATED_SOURCE_COLUMNS = {
    "tpep_pickup_datetime",
    "lpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "lpep_dropoff_datetime",
    "pulocationid",
    "dolocationid",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "tip_amount",
    "tolls_amount",
    "ratecodeid",
    "sr_flag",
    "vendorid",
    "taxi_type",
    "pickup_day",
    "pickup_year",
    "trip_duration_minutes",
}

## 4. Deteccion de datasets limpios

In [5]:
cleaned_datasets = sorted(
    path
    for path in CLEANED_BASE_DIR.glob("*/*/*_cleaned.parquet")
    if path.is_dir() or path.is_file()
)

if not cleaned_datasets:
    raise FileNotFoundError(f"No se encontraron datasets cleaned en {CLEANED_BASE_DIR}")

for dataset_path in cleaned_datasets:
    taxi_type, year = parse_cleaned_dataset_path(dataset_path)
    print(f"- {dataset_path.relative_to(PROJECT_ROOT)}: tipo={taxi_type}, anio={year}")

- data\silver\cleaned\fhv\year=2023\fhv_2023_cleaned.parquet: tipo=fhv, anio=2023
- data\silver\cleaned\fhv\year=2024\fhv_2024_cleaned.parquet: tipo=fhv, anio=2024
- data\silver\cleaned\fhv\year=2025\fhv_2025_cleaned.parquet: tipo=fhv, anio=2025
- data\silver\cleaned\green\year=2023\green_2023_cleaned.parquet: tipo=green, anio=2023
- data\silver\cleaned\green\year=2024\green_2024_cleaned.parquet: tipo=green, anio=2024
- data\silver\cleaned\green\year=2025\green_2025_cleaned.parquet: tipo=green, anio=2025
- data\silver\cleaned\yellow\year=2023\yellow_2023_cleaned.parquet: tipo=yellow, anio=2023
- data\silver\cleaned\yellow\year=2024\yellow_2024_cleaned.parquet: tipo=yellow, anio=2024
- data\silver\cleaned\yellow\year=2025\yellow_2025_cleaned.parquet: tipo=yellow, anio=2025


## 5. Normalizacion

In [6]:
def normalize_trip_dataset(dataset_path: Path) -> dict:
    taxi_type, year = parse_cleaned_dataset_path(dataset_path)
    output_path = OUTPUT_BASE_DIR / f"{taxi_type}_{year}.parquet"
    started_at = datetime.now()
    created_columns: list[str] = []

    print(f"\nProcesando {dataset_path.relative_to(PROJECT_ROOT)}...")
    input_files = parquet_part_files(dataset_path)
    if not input_files:
        raise FileNotFoundError(f"No se encontraron archivos .parquet dentro de {dataset_path}")

    df = normalize_columns(spark.read.parquet(*input_files))
    original_columns = df.columns[:]
    input_records = count_records(df)

    pickup_expr, pickup_source = column_expr_or_null(df, PICKUP_DATETIME_CANDIDATES, "timestamp")
    dropoff_expr, dropoff_source = column_expr_or_null(df, DROPOFF_DATETIME_CANDIDATES, "timestamp")
    pickup_location_expr, pickup_location_source = column_expr_or_null(df, PICKUP_LOCATION_CANDIDATES, "int")
    dropoff_location_expr, dropoff_location_source = column_expr_or_null(df, DROPOFF_LOCATION_CANDIDATES, "int")
    distance_expr, distance_source = column_expr_or_null(df, DISTANCE_CANDIDATES, "double")
    base_amount_expr, base_amount_source = column_expr_or_null(df, BASE_AMOUNT_CANDIDATES, "double")
    total_amount_expr, total_amount_source = column_expr_or_null(df, TOTAL_AMOUNT_CANDIDATES, "double")
    tip_expr, tip_source = column_expr_or_null(df, TIP_CANDIDATES, "double")
    tolls_expr, tolls_source = column_expr_or_null(df, TOLLS_CANDIDATES, "double")
    payment_expr, payment_source = column_expr_or_null(df, PAYMENT_CANDIDATES, "int")
    passenger_expr, passenger_source = column_expr_or_null(df, PASSENGER_CANDIDATES, "double")
    vendor_expr, vendor_source = column_expr_or_null(df, VENDOR_CANDIDATES, "string")
    ratecode_expr, ratecode_source = column_expr_or_null(df, RATECODE_CANDIDATES, "int")
    shared_ride_expr, shared_ride_source = column_expr_or_null(df, SHARED_RIDE_CANDIDATES, "int")

    df = ensure_canonical_column(df, "tipo_dataset", F.lit(taxi_type), created_columns)
    df = ensure_canonical_column(df, "anio", F.lit(year).cast("int"), created_columns)
    df = ensure_canonical_column(df, "pickup_datetime", pickup_expr, created_columns)
    df = ensure_canonical_column(df, "dropoff_datetime", dropoff_expr, created_columns)
    df = ensure_canonical_column(df, "pickup_location_id", pickup_location_expr, created_columns)
    df = ensure_canonical_column(df, "dropoff_location_id", dropoff_location_expr, created_columns)
    df = ensure_canonical_column(df, "distancia_millas", distance_expr, created_columns)
    df = ensure_canonical_column(df, "monto_base", base_amount_expr, created_columns)
    df = ensure_canonical_column(df, "monto_total", total_amount_expr, created_columns)
    df = ensure_canonical_column(df, "propina", tip_expr, created_columns)
    df = ensure_canonical_column(df, "peajes", tolls_expr, created_columns)
    df = ensure_canonical_column(df, "payment_type", payment_expr, created_columns)
    df = ensure_canonical_column(df, "passenger_count", passenger_expr, created_columns)
    df = ensure_canonical_column(df, "vendor_id", vendor_expr, created_columns)
    df = ensure_canonical_column(df, "ratecode_id", ratecode_expr, created_columns)
    df = ensure_canonical_column(df, "shared_ride_flag", shared_ride_expr, created_columns)

    for canonical_name, candidates in OPTIONAL_DOUBLE_COLUMNS.items():
        expr, _ = column_expr_or_null(df, candidates, "double")
        df = ensure_canonical_column(df, canonical_name, expr, created_columns)

    df = df.withColumn("pickup_datetime", F.to_timestamp(F.col("pickup_datetime")))
    df = df.withColumn("dropoff_datetime", F.to_timestamp(F.col("dropoff_datetime")))
    if "duracion_segundos" not in df.columns:
        created_columns.append("duracion_segundos")
        df = df.withColumn(
            "duracion_segundos",
            F.when(
                F.col("pickup_datetime").isNotNull() & F.col("dropoff_datetime").isNotNull(),
                F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime"),
            ).cast("long"),
        )
    if "duracion_minutos" not in df.columns:
        created_columns.append("duracion_minutos")
        df = df.withColumn("duracion_minutos", (F.col("duracion_segundos") / F.lit(60.0)).cast("double"))

    df = ensure_canonical_column(df, "pickup_date", F.to_date("pickup_datetime"), created_columns)
    df = ensure_canonical_column(df, "pickup_hour", F.hour("pickup_datetime"), created_columns)
    df = ensure_canonical_column(df, "pickup_day_of_week", F.dayofweek("pickup_datetime"), created_columns)
    df = ensure_canonical_column(df, "pickup_month", F.month("pickup_datetime"), created_columns)

    for numeric_column in [
        "pickup_location_id",
        "dropoff_location_id",
        "distancia_millas",
        "monto_base",
        "monto_total",
        "propina",
        "peajes",
        "extra",
        "mta_tax",
        "improvement_surcharge",
        "congestion_surcharge",
        "airport_fee",
        "cbd_congestion_fee",
        "ehail_fee",
        "payment_type",
        "passenger_count",
        "ratecode_id",
        "trip_type",
        "shared_ride_flag",
    ]:
        if numeric_column in df.columns:
            target_type = "int" if numeric_column in {"pickup_location_id", "dropoff_location_id", "payment_type", "ratecode_id", "trip_type", "shared_ride_flag"} else "double"
            df = df.withColumn(numeric_column, F.col(numeric_column).cast(target_type))

    # Salida ligera: conserva lo necesario para Gold/Power BI y evita duplicados pesados.
    ordered_columns = [column for column in CANONICAL_ORDER if column in df.columns]
    remaining_columns = [
        column
        for column in ESSENTIAL_EXTRA_COLUMNS
        if column in df.columns and column not in ordered_columns and column not in DUPLICATED_SOURCE_COLUMNS
    ]
    df = df.select(*ordered_columns, *remaining_columns)

    output_path.mkdir(parents=True, exist_ok=True)
    df.write.mode("overwrite").parquet(str(output_path))

    record = {
        "status": "completed",
        "dataset": f"{taxi_type}_{year}.parquet",
        "taxi_type": taxi_type,
        "year": year,
        "input_path": str(dataset_path),
        "input_files": len(input_files),
        "output_path": str(output_path),
        "input_records": input_records,
        "output_records": input_records,
        "original_columns": len(original_columns),
        "output_columns": len(df.columns),
        "created_columns": sorted(set(created_columns)),
        "source_columns": {
            "pickup_datetime": pickup_source,
            "dropoff_datetime": dropoff_source,
            "pickup_location_id": pickup_location_source,
            "dropoff_location_id": dropoff_location_source,
            "distancia_millas": distance_source,
            "monto_base": base_amount_source,
            "monto_total": total_amount_source,
            "propina": tip_source,
            "peajes": tolls_source,
            "payment_type": payment_source,
            "passenger_count": passenger_source,
            "vendor_id": vendor_source,
            "ratecode_id": ratecode_source,
            "shared_ride_flag": shared_ride_source,
        },
        "duration_seconds": round((datetime.now() - started_at).total_seconds(), 2),
    }
    append_audit(record)
    print(f"Guardado: {output_path}")
    print(f"Registros conservados: {input_records:,}")
    print(f"Columnas creadas: {', '.join(record['created_columns'])}")

    del df
    return record

## 6. Ejecucion masiva y auditoria

In [7]:
# Deja la lista vacia para procesar todos los datasets detectados.
# Ejemplo de filtro: DATASETS_TO_PROCESS = [("yellow", 2025), ("green", 2025)]
DATASETS_TO_PROCESS: list[tuple[str, int]] = []

selected_datasets = []
selected_keys = set(DATASETS_TO_PROCESS)
for dataset_path in cleaned_datasets:
    taxi_type, year = parse_cleaned_dataset_path(dataset_path)
    if not selected_keys or (taxi_type, year) in selected_keys:
        selected_datasets.append(dataset_path)

if not selected_datasets:
    raise ValueError("No hay datasets seleccionados para normalizar.")

normalization_results = []
for dataset_path in selected_datasets:
    taxi_type, year = parse_cleaned_dataset_path(dataset_path)
    try:
        normalization_results.append(normalize_trip_dataset(dataset_path))
    except Exception as error:
        error_record = {
            "status": "failed",
            "dataset": f"{taxi_type}_{year}.parquet",
            "taxi_type": taxi_type,
            "year": year,
            "input_path": str(dataset_path),
            "error": str(error),
        }
        append_audit(error_record)
        print(f"ERROR en {dataset_path.name}: {error}")
        raise
    finally:
        release_spark_memory()

print("\nResumen de normalizacion:")
for result in normalization_results:
    print(
        f"- {result['dataset']}: {result['input_records']:,} registros, "
        f"{result['output_columns']} columnas, {result['duration_seconds']} s"
    )


Procesando data\silver\cleaned\fhv\year=2023\fhv_2023_cleaned.parquet...
Guardado: C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos\data\silver\trip_data_normalized\fhv_2023.parquet
Registros conservados: 15,519,045
Columnas creadas: airport_fee, anio, cbd_congestion_fee, congestion_surcharge, distancia_millas, dropoff_location_id, duracion_minutos, duracion_segundos, ehail_fee, extra, improvement_surcharge, monto_base, monto_total, mta_tax, passenger_count, payment_type, peajes, pickup_day_of_week, pickup_location_id, propina, ratecode_id, shared_ride_flag, tipo_dataset, vendor_id

Procesando data\silver\cleaned\fhv\year=2024\fhv_2024_cleaned.parquet...
Guardado: C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos\data\silver\trip_data_normalized\fhv_2024.parquet
Registros conservados: 17,175,195
Columnas creadas: airport_fee, anio, cbd_congestion

## 7. Validacion rapida de salida

In [8]:
normalized_datasets = sorted(path for path in OUTPUT_BASE_DIR.iterdir() if path.is_dir() and path.name.endswith(".parquet"))
print(f"Datasets normalizados disponibles: {len(normalized_datasets)}")
for dataset_path in normalized_datasets:
    print(f"- {dataset_path.name}")

sample_path = normalized_datasets[0]
sample_files = parquet_part_files(sample_path)
sample_df = spark.read.parquet(*sample_files)
print(f"\nMuestra de esquema: {sample_path.name}")
sample_df.select(*[column for column in CANONICAL_ORDER if column in sample_df.columns]).printSchema()
sample_df.select(*[column for column in CANONICAL_ORDER if column in sample_df.columns]).show(5, truncate=False)

Datasets normalizados disponibles: 9
- fhv_2023.parquet
- fhv_2024.parquet
- fhv_2025.parquet
- green_2023.parquet
- green_2024.parquet
- green_2025.parquet
- yellow_2023.parquet
- yellow_2024.parquet
- yellow_2025.parquet

Muestra de esquema: fhv_2023.parquet
root
 |-- tipo_dataset: string (nullable = true)
 |-- anio: integer (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- pickup_location_id: integer (nullable = true)
 |-- dropoff_location_id: integer (nullable = true)
 |-- distancia_millas: double (nullable = true)
 |-- duracion_segundos: long (nullable = true)
 |-- duracion_minutos: double (nullable = true)
 |-- monto_base: double (nullable = true)
 |-- monto_total: double (nullable = true)
 |-- propina: double (nullable = true)
 |-- peajes: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- congestion